# Random Forest and Logistic Regression Classification Notebook

This notebook:
1. Creates a synthetic classification dataset.
2. Splits the dataset into train, validation, and test sets using a 70%, 15%, 15% ratio.
3. Trains Random Forest and Logistic Regression models.
4. Evaluates both models.
5. Generates predictions and saves them into CSV files.


In [ ]:
# Import NumPy for numerical operations.
import numpy as np

# Import Pandas for creating and handling tabular datasets.
import pandas as pd

# Import make_classification to generate a synthetic classification dataset.
from sklearn.datasets import make_classification

# Import train_test_split to split the dataset into train, validation, and test sets.
from sklearn.model_selection import train_test_split

# Import StandardScaler to scale features for Logistic Regression.
from sklearn.preprocessing import StandardScaler

# Import RandomForestClassifier for the Random Forest model.
from sklearn.ensemble import RandomForestClassifier

# Import LogisticRegression for the Logistic Regression model.
from sklearn.linear_model import LogisticRegression

# Import accuracy_score to calculate classification accuracy.
from sklearn.metrics import accuracy_score

# Import precision_score to calculate precision.
from sklearn.metrics import precision_score

# Import recall_score to calculate recall.
from sklearn.metrics import recall_score

# Import f1_score to calculate F1-score.
from sklearn.metrics import f1_score

# Import classification_report to generate a detailed classification report.
from sklearn.metrics import classification_report

# Import confusion_matrix to generate the confusion matrix.
from sklearn.metrics import confusion_matrix

# Set a random seed so that results are reproducible.
RANDOM_STATE = 42


## 1. Create Synthetic Dataset

In [ ]:
# Create a synthetic binary classification dataset.
X, y = make_classification(
    n_samples=1000,              # Define the total number of data samples.
    n_features=10,               # Define the total number of input features.
    n_informative=6,             # Define how many features are useful for prediction.
    n_redundant=2,               # Define how many features are redundant combinations of useful features.
    n_repeated=0,                # Define how many repeated features should be created.
    n_classes=2,                 # Define the number of output classes.
    weights=[0.55, 0.45],        # Define class balance for class 0 and class 1.
    class_sep=1.2,               # Define how separable the classes are.
    random_state=RANDOM_STATE    # Use the fixed random seed for reproducibility.
)

# Create column names for the feature columns.
feature_names = [f"feature_{i+1}" for i in range(X.shape[1])]

# Convert the feature array into a Pandas DataFrame.
df = pd.DataFrame(X, columns=feature_names)

# Add the target column to the DataFrame.
df["target"] = y

# Display the first five rows of the dataset.
df.head()


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,target
0,-1.230931,1.591626,0.747274,0.928932,-1.938880,1.509549,1.332551,1.778256,2.491179,-0.518434,0
1,-2.966254,1.447870,-0.503691,1.083145,0.910836,1.907480,-1.794192,2.546422,1.686321,0.198810,1
2,-0.758987,0.499849,1.727071,0.360442,-1.560209,1.360340,-0.755951,1.531933,2.407562,-1.024404,0
3,-1.550289,-2.246078,-0.814264,0.126459,-0.983923,6.525685,-0.915477,-3.384768,-0.364542,-4.120960,0
4,-0.475754,-0.928495,-0.172273,-0.660834,-2.128161,4.175604,1.446944,-1.311662,0.348484,-2.576528,0


In [ ]:
# Print the shape of the full dataset.
print("Dataset shape:", df.shape)

# Print the class distribution of the target variable.
print("\nClass distribution:")

# Display how many samples belong to each class.
print(df["target"].value_counts())


Dataset shape: (1000, 11)

Class distribution:
target
0    548
1    452
Name: count, dtype: int64


## 2. Split Dataset into 70%, 15%, 15%

In [ ]:
# Separate input features from the target column.
X = df.drop("target", axis=1)

# Store the target column separately.
y = df["target"]

# First split the data into 70% training and 30% temporary data.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,                         # Input features.
    y,                         # Target labels.
    test_size=0.30,            # Keep 30% for validation and testing.
    random_state=RANDOM_STATE, # Use fixed random seed.
    stratify=y                 # Preserve class distribution in the split.
)

# Split the temporary 30% data equally into 15% validation and 15% test data.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,                    # Temporary input features.
    y_temp,                    # Temporary target labels.
    test_size=0.50,            # Half of 30% becomes test data.
    random_state=RANDOM_STATE, # Use fixed random seed.
    stratify=y_temp            # Preserve class distribution in validation and test sets.
)

# Print the number of rows in the training set.
print("Training set size:", X_train.shape[0])

# Print the number of rows in the validation set.
print("Validation set size:", X_val.shape[0])

# Print the number of rows in the test set.
print("Test set size:", X_test.shape[0])


Training set size: 700
Validation set size: 150
Test set size: 150


## 3. Feature Scaling for Logistic Regression

Random Forest does not require feature scaling.  
Logistic Regression usually performs better when the features are scaled.


In [ ]:
# Create a StandardScaler object.
scaler = StandardScaler()

# Fit the scaler only on the training features and transform the training features.
X_train_scaled = scaler.fit_transform(X_train)

# Transform the validation features using the scaler fitted on training data.
X_val_scaled = scaler.transform(X_val)

# Transform the test features using the scaler fitted on training data.
X_test_scaled = scaler.transform(X_test)


## 4. Train Random Forest Model
Main Idea behind Random Forest Model:

Instead of depending on only one decision tree, Random Forest creates:

* multiple trees
* each tree learns from slightly different data
* final prediction is based on voting or averaging

This reduces overfitting and improves accuracy.

In [ ]:
# Create a Random Forest classifier object.
rf_model = RandomForestClassifier(
    n_estimators=100,           # Use 100 decision trees.
    max_depth=None,             # Allow trees to grow until stopping criteria are met.
    random_state=RANDOM_STATE,  # Use fixed random seed.
    class_weight="balanced"     # Handle possible class imbalance.
)

# Train the Random Forest model using the training data.
rf_model.fit(X_train, y_train)

# Generate predictions on the validation set using Random Forest.
rf_val_predictions = rf_model.predict(X_val)

# Generate predictions on the test set using Random Forest.
rf_test_predictions = rf_model.predict(X_test)


## 5. Train Logistic Regression Model

Main Idea behind Logistic Regression:

It predicts the probability of belonging to a class. Output is always between: 0 and 1. Then:

* Probability > 0.5 -> Class 1
* Probability < 0.5 -> Class 0

In [ ]:
# Create a Logistic Regression classifier object.
lr_model = LogisticRegression(
    max_iter=1000,              # Allow enough iterations for convergence.
    random_state=RANDOM_STATE,  # Use fixed random seed.
    class_weight="balanced"     # Handle possible class imbalance.
)

# Train the Logistic Regression model using the scaled training data.
lr_model.fit(X_train_scaled, y_train)

# Generate predictions on the validation set using Logistic Regression.
lr_val_predictions = lr_model.predict(X_val_scaled)

# Generate predictions on the test set using Logistic Regression.
lr_test_predictions = lr_model.predict(X_test_scaled)


## 6. Evaluation Function

In [ ]:
# Define a function to evaluate a classification model.
def evaluate_model(model_name, y_true, y_pred):
    # Print the model name.
    print(f"\nModel: {model_name}")

    # Calculate model accuracy.
    accuracy = accuracy_score(y_true, y_pred)

    # Calculate model precision.
    precision = precision_score(y_true, y_pred)

    # Calculate model recall.
    recall = recall_score(y_true, y_pred)

    # Calculate model F1-score.
    f1 = f1_score(y_true, y_pred)

    # Print the accuracy score.
    print("Accuracy:", round(accuracy, 4))

    # Print the precision score.
    print("Precision:", round(precision, 4))

    # Print the recall score.
    print("Recall:", round(recall, 4))

    # Print the F1-score.
    print("F1-score:", round(f1, 4))

    # Print the confusion matrix.
    print("\nConfusion Matrix:")

    # Display the confusion matrix.
    print(confusion_matrix(y_true, y_pred))

    # Print the full classification report.
    print("\nClassification Report:")

    # Display precision, recall, F1-score, and support for each class.
    print(classification_report(y_true, y_pred))


## 7. Evaluate on Validation Set

In [ ]:
# Evaluate Random Forest on the validation set.
evaluate_model("Random Forest - Validation", y_val, rf_val_predictions)

# Evaluate Logistic Regression on the validation set.
evaluate_model("Logistic Regression - Validation", y_val, lr_val_predictions)



Model: Random Forest - Validation
Accuracy: 0.9533
Precision: 0.9067
Recall: 1.0
F1-score: 0.951

Confusion Matrix:
[[75  7]
 [ 0 68]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.91      0.96        82
           1       0.91      1.00      0.95        68

    accuracy                           0.95       150
   macro avg       0.95      0.96      0.95       150
weighted avg       0.96      0.95      0.95       150


Model: Logistic Regression - Validation
Accuracy: 0.86
Precision: 0.7975
Recall: 0.9265
F1-score: 0.8571

Confusion Matrix:
[[66 16]
 [ 5 63]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.80      0.86        82
           1       0.80      0.93      0.86        68

    accuracy                           0.86       150
   macro avg       0.86      0.87      0.86       150
weighted avg       0.87      0.86      0.86       150



## 8. Evaluate on Test Set

In [ ]:
# Evaluate Random Forest on the test set.
evaluate_model("Random Forest - Test", y_test, rf_test_predictions)

# Evaluate Logistic Regression on the test set.
evaluate_model("Logistic Regression - Test", y_test, lr_test_predictions)



Model: Random Forest - Test
Accuracy: 0.9333
Precision: 0.9028
Recall: 0.9559
F1-score: 0.9286

Confusion Matrix:
[[75  7]
 [ 3 65]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.91      0.94        82
           1       0.90      0.96      0.93        68

    accuracy                           0.93       150
   macro avg       0.93      0.94      0.93       150
weighted avg       0.93      0.93      0.93       150


Model: Logistic Regression - Test
Accuracy: 0.86
Precision: 0.8133
Recall: 0.8971
F1-score: 0.8531

Confusion Matrix:
[[68 14]
 [ 7 61]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.83      0.87        82
           1       0.81      0.90      0.85        68

    accuracy                           0.86       150
   macro avg       0.86      0.86      0.86       150
weighted avg       0.86      0.86      0.86       150



## 9. Generate Prediction Tables

In [ ]:
# Create a prediction DataFrame for the Random Forest model.
rf_predictions_df = X_test.copy()

# Add the actual target values to the Random Forest prediction DataFrame.
rf_predictions_df["actual_target"] = y_test.values

# Add the Random Forest predicted target values.
rf_predictions_df["predicted_target"] = rf_test_predictions

# Add the Random Forest predicted probability for class 1.
rf_predictions_df["predicted_probability_class_1"] = rf_model.predict_proba(X_test)[:, 1]

# Display the first five Random Forest predictions.
rf_predictions_df.head()


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,actual_target,predicted_target,predicted_probability_class_1
70,-0.990244,-1.369568,2.564622,-1.339470,-2.272591,2.250554,-1.481581,1.517157,3.599186,-0.920171,0,0,0.06
470,-1.418842,0.361141,-0.056040,2.099408,-0.916842,0.466706,1.352039,-0.740885,2.185640,0.646845,1,1,0.89
862,-3.309958,1.936733,-2.381867,-0.565259,1.322759,0.394287,-2.385114,-1.330526,1.371744,1.583338,1,1,0.98
669,-3.023423,1.776825,-1.825417,0.544582,1.074167,0.937872,-2.850030,-1.282589,1.288627,0.671038,1,1,0.88
407,-3.081148,0.279542,-0.264961,-0.120587,1.084560,1.233953,1.330432,-0.087432,2.337826,0.570901,1,1,0.69


In [ ]:
# Create a prediction DataFrame for the Logistic Regression model.
lr_predictions_df = X_test.copy()

# Add the actual target values to the Logistic Regression prediction DataFrame.
lr_predictions_df["actual_target"] = y_test.values

# Add the Logistic Regression predicted target values.
lr_predictions_df["predicted_target"] = lr_test_predictions

# Add the Logistic Regression predicted probability for class 1.
lr_predictions_df["predicted_probability_class_1"] = lr_model.predict_proba(X_test_scaled)[:, 1]

# Display the first five Logistic Regression predictions.
lr_predictions_df.head()


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,actual_target,predicted_target,predicted_probability_class_1
70,-0.990244,-1.369568,2.564622,-1.339470,-2.272591,2.250554,-1.481581,1.517157,3.599186,-0.920171,0,0,0.033886
470,-1.418842,0.361141,-0.056040,2.099408,-0.916842,0.466706,1.352039,-0.740885,2.185640,0.646845,1,1,0.692399
862,-3.309958,1.936733,-2.381867,-0.565259,1.322759,0.394287,-2.385114,-1.330526,1.371744,1.583338,1,1,0.792298
669,-3.023423,1.776825,-1.825417,0.544582,1.074167,0.937872,-2.850030,-1.282589,1.288627,0.671038,1,1,0.579159
407,-3.081148,0.279542,-0.264961,-0.120587,1.084560,1.233953,1.330432,-0.087432,2.337826,0.570901,1,1,0.593673


## 10. Save Dataset and Predictions

In [ ]:
# Save the full generated dataset into a CSV file.
df.to_csv("generated_classification_dataset.csv", index=False)

# Save the Random Forest predictions into a CSV file.
rf_predictions_df.to_csv("random_forest_predictions.csv", index=False)

# Save the Logistic Regression predictions into a CSV file.
lr_predictions_df.to_csv("logistic_regression_predictions.csv", index=False)

# Print a confirmation message.
print("Files saved successfully.")


## 11. Predict a New Sample

In [ ]:
# Select one sample from the test set as a new input example.
new_sample = X_test.iloc[[0]]

# Scale the new sample for Logistic Regression.
new_sample_scaled = scaler.transform(new_sample)

# Predict the class of the new sample using Random Forest.
rf_new_prediction = rf_model.predict(new_sample)

# Predict the probability of class 1 using Random Forest.
rf_new_probability = rf_model.predict_proba(new_sample)[:, 1]

# Predict the class of the new sample using Logistic Regression.
lr_new_prediction = lr_model.predict(new_sample_scaled)

# Predict the probability of class 1 using Logistic Regression.
lr_new_probability = lr_model.predict_proba(new_sample_scaled)[:, 1]

# Print the Random Forest prediction.
print("Random Forest predicted class:", rf_new_prediction[0])

# Print the Random Forest predicted probability.
print("Random Forest probability for class 1:", round(rf_new_probability[0], 4))

# Print the Logistic Regression prediction.
print("Logistic Regression predicted class:", lr_new_prediction[0])

# Print the Logistic Regression predicted probability.
print("Logistic Regression probability for class 1:", round(lr_new_probability[0], 4))


Random Forest predicted class: 0
Random Forest probability for class 1: 0.06
Logistic Regression predicted class: 0
Logistic Regression probability for class 1: 0.0339
